### Building A Chatbot

#### In this We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.


#### Note: This chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

* Conversational RAG: Enable a chatbot experience over an external source of data
* Agents: Build a chatbot that can take actions

will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")
llm.invoke("tell me a joke")

AIMessage(content='A man walked into a library and asked the librarian, "Do you have any books on Pavlov\'s dogs and Schrödinger\'s cat?" \n\nThe librarian replied, "It rings a bell, but I\'m not sure if it\'s here or not."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 39, 'total_tokens': 94, 'completion_time': 0.057080271, 'completion_tokens_details': None, 'prompt_time': 0.002482889, 'prompt_tokens_details': None, 'queue_time': 0.045873331, 'total_time': 0.05956316}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0ac4-312e-7d61-805f-5044d74ac6e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 55, 'total_tokens': 94})

In [45]:
from langchain_core.messages import HumanMessage,AIMessage

# HumanMessage means user is giving input
# AI Message means llm is giving output based on the Human Message

result = llm.invoke([HumanMessage(content="Hi My name is leo, im planning to become Chief AI Engineer")])
result

AIMessage(content="Hello Leo, it's great to meet you. Becoming a Chief AI Engineer is a highly ambitious and exciting goal. To help you get started, I'll need to know a bit more about your current situation and goals. Can you please share the following information with me:\n\n1. What's your current level of experience in AI and engineering?\n2. Do you have a degree in a relevant field, such as computer science, mathematics, or engineering?\n3. What specific areas of AI are you interested in (e.g., machine learning, natural language processing, computer vision)?\n4. What kind of company or organization do you hope to work for (e.g., tech startup, large corporation, research institution)?\n5. Are there any specific skills or certifications you're looking to acquire or improve upon?\n\nOnce I have this information, I can provide you with more tailored guidance and support to help you achieve your goal of becoming a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage

In [46]:
result_new = llm.invoke(
    [
    HumanMessage(content="Hi My name is leo, im planning to become Chief AI Engineer"),
    AIMessage(content="That's a fascinating and in-demand career goal. Becoming a Chief AI Engineer requires a strong foundation in both AI and engineering, as well as leadership and management skills"),
    HumanMessage(content="Hey What do im planning to do?")]
    )


In [47]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
output_parser.invoke(result_new)

"You're planning to become a Chief AI Engineer, which means you'll be responsible for leading and overseeing the development and implementation of Artificial Intelligence (AI) systems in organizations. Your role will likely involve:\n\n1. **Strategic Planning**: Developing and executing AI strategies to drive business growth and innovation.\n2. **Technical Leadership**: Overseeing the design, development, and deployment of AI systems, including natural language processing, computer vision, and machine learning.\n3. **Team Management**: Leading and mentoring a team of AI engineers, data scientists, and researchers to ensure the delivery of high-quality AI solutions.\n4. **Collaboration**: Working closely with cross-functional teams, including business stakeholders, to identify AI opportunities and implement solutions.\n5. **Staying Up-to-Date**: Continuously learning about new AI technologies, techniques, and best practices to stay ahead of the curve.\n\nTo achieve this goal, you'll nee

### Message History --> Most important 

#### We can use a Message History Class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model and store them in some datastore. Future Interactions will load those messages and pass them into the chain as part of the input. Lets see how to use this

In [48]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

#Question: Many users are chatting with llm's how can we make every session unique to that person?

store = {}

# This function will create a session id
def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id] 

with_message_history = RunnableWithMessageHistory(llm,get_session_history)

In [39]:
config={"configurable":{"session_id":"chat1"}}

In [49]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi My name is leo, im planning to become Chief AI Engineer")],
    config=config
)


In [50]:
response.content

'Nice to meet you, Leo. Becoming a Chief AI Engineer is an exciting and challenging goal. It requires a strong foundation in computer science, mathematics, and AI-related fields. Here are some steps and recommendations to help you achieve your goal:\n\n1. **Develop a strong foundation in computer science**: Focus on programming languages such as Python, Java, and C++. Understand data structures, algorithms, and software engineering principles.\n2. **Learn mathematics**: Mathematics is a crucial component of AI. Study linear algebra, calculus, probability, and statistics.\n3. **Study AI and ML fundamentals**: Learn the basics of machine learning, deep learning, and natural language processing. Familiarize yourself with popular libraries and frameworks like TensorFlow, PyTorch, and scikit-learn.\n4. **Get hands-on experience**: Work on projects that involve building AI models, deploying them on cloud platforms, and integrating them with other systems.\n5. **Stay up-to-date with industry 

In [51]:
response_new = with_message_history.invoke(
    [HumanMessage(content="what is my name")],
    config=config
)
response_new.content

'Your name is Leo.'

In [56]:
## Changing the config which means session_id different.

config1={"configurable":{"session_id":"chat2"}}
response_2 = with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config1
)
response_2.content

'You told me a little while ago, Davinci. Your name is Davinci.'

In [ ]:
response_2 = with_message_history.invoke(
    [HumanMessage(content="Hi My name is davinci")],
    config=config1
)
response_2.content

"Nice to meet you, Davinci. You're a unique name, and I'm guessing it might be inspired by the famous artist Leonardo da Vinci. Am I correct?"

## Prompt templates

### Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated.

### First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
## Add more complexity
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder # instead of messages we can use MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",'you are helpful assisstant.Answer all the question to the best of your ability in'),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | llm

In [59]:
chain.invoke({"messages":[HumanMessage(content="Hi My Name is Da Vinci")]})

AIMessage(content="Welcome, Maestro Da Vinci. It's an honor to converse with the renowned Italian polymath and one of the greatest minds in human history. You are famous for your inventions, art, science, mathematics, and anatomy. Your works continue to inspire and fascinate people around the world.\n\nPlease, tell me what brings you here today? Do you have a question, a problem to solve, or perhaps a new idea you'd like to share? I'm all ears, Maestro.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 58, 'total_tokens': 157, 'completion_time': 0.16000594, 'completion_tokens_details': None, 'prompt_time': 0.002804365, 'prompt_tokens_details': None, 'queue_time': 0.046004395, 'total_time': 0.162810305}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0bea-4d03-7c52-a95c-345cd90f239d-0', tool_ca

In [60]:
with_message_history_latest = RunnableWithMessageHistory(chain,get_session_history)

In [62]:
config = {"configurable": {"session_id":"chat3"}}

response = with_message_history_latest.invoke(
    [HumanMessage(content="Hi My Name is Da Vinci")],
    config = config
)

response.content

"As I mentioned earlier, the famous artist and inventor was named Leonardo da Vinci. But if you're referring to yourself as Da Vinci, that's perfectly fine.\n\nIn that case, hello Da Vinci! What brings you here today? What are your interests or hobbies? I'm here to listen and help with any questions or topics you'd like to discuss."

In [63]:
### Add More complexity

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",'you are helpful assisstant.Answer all the question to the best of your ability in {language}'),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | llm

In [64]:
respone = chain.invoke({"messages":[HumanMessage(content="Hi My name is leonardo")],"language":"Hindi"})
respone.content

'नमस्ते लियोनार्डो, मैं आपकी सहायता के लिए यहाँ हूँ। क्या आपके पास कोई प्रश्न है?'

Lets now wrap this more complicated chain in Message History Class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [65]:
with_message_history_latest_new = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

with_message_history_latest_new

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  messages: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x0000020DAB6CE7A0>, input_messages_key='messages', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [ ]:
config = {"configurable": {"session_id":"chat4"}}

response = with_message_history_latest_new.invoke(
    {
    "messages": [HumanMessage(content="Hi, Im Davinci")],"language":"Hindi"
    },
    config = config
)


In [ ]:
response.content

'नमस्ते डाविन्ची जी! (Namaste Davinci ji!) आप कैसे हैं? मैं आपकी सहायता के लिए यहाँ हूँ। आपके पास कोई सवाल या कुछ जानने की इच्छा है?'

In [73]:
response = with_message_history_latest_new.invoke(
    {
    "messages": [HumanMessage(content="What is my name")],"language":"English"
    },
    config = config
)


In [74]:
response.content

'Your name is Davinci.'

### Manage the Conversation History

- one Important Concept to understand when building chatbots is how to mange conversation history.
- If Unmanaged, The list of messages will grow unbounded and potentially overflow the context window of the LLM.
- Therefore, It is important to add a step that limits the size of the messages you are passing in

* "trim_messages" : Helper to reduce how many messages we are sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other Parameters like if we want to always keep the system message and whether to allow partial messages

In [79]:
from langchain_core.messages import SystemMessage,trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy="last", # Focus more on last message
    token_counter=llm,
    include_system=True,
    allow_partial=False,
    start_on="human"
    )

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="Hi! I'm Bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like Vanilla Icecream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="Having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Hi! I'm Bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like Vanilla Icecream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs=

In [80]:
### Use chain to pass the trimmer function

from operator import  itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer)
    | prompt
    | llm
)

chain.invoke({
    "messages": messages + [HumanMessage(content="What icecream do i like")],
    "language" : 'English'
}
)

AIMessage(content='Vanilla ice cream!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 146, 'total_tokens': 152, 'completion_time': 0.013720004, 'completion_tokens_details': None, 'prompt_time': 0.008481695, 'prompt_tokens_details': None, 'queue_time': 0.046038765, 'total_time': 0.022201699}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0c1d-51ee-7d63-8efc-182840b67e2f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 146, 'output_tokens': 6, 'total_tokens': 152})

In [82]:
# Lets Wrap this in the Message History

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable":{"session_id":"chat5"}}

In [83]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="Whats my name?")],
        "language": "English",
    },
    config = config
)

response.content

'Bob'